# OOT Investigation Workbench — Demo

End-to-end demonstration of the investigation pipeline:
1. Generate / load the dataset
2. Build the feature store
3. Detect OOT/OOS events
4. Run a peer-batch comparison for a suspect lot
5. Rank contributing factors
6. Generate an investigation memo

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve() / '..' / 'src'))

from data.pull_public_dataset import generate_realistic_dataset
from features.build_feature_store import build_feature_store, get_modeling_columns
from analytics.detect_oot_events import detect_oot_events, summarize_suspects
from analytics.peer_batch_compare import select_peer_lots, compute_comparisons, top_differences
from models.rank_contributors import prepare_labels, train_contributor_model, rank_contributors
from reports.generate_investigation_report import generate_investigation_report

## 1. Generate the dataset

In [2]:
tables = generate_realistic_dataset(n_batches=1005, seed=42)
print(f"Laboratory: {tables['laboratory'].shape}")
print(f"Process: {tables['process'].shape}")
print(f"Deviations: {tables['deviation_log'].shape}")
tables['laboratory'].head()

Laboratory: (1005, 16)
Process: (1005, 9)
Deviations: (80, 4)


,batch_id,product_code,batch_size_category,mfg_date,rm_potency,rm_moisture,rm_particle_size,rm_purity,rm_supplier,dissolution,hardness,friability,disintegration,assay,content_uniformity_rsd,water_content
0,LOT-0001,PROD_B,standard,2022-01-05 00:00:00,101.46,2.91,52.37,99.68,SUP_1,81.45,7.96,0.742,10.8,102.07,1.62,2.12
1,LOT-0002,PROD_A,large,2022-01-05 08:00:00,100.72,2.48,52.50,99.56,SUP_2,84.61,8.16,0.555,14.2,100.73,2.31,1.87
2,LOT-0003,PROD_C,standard,2022-01-05 16:00:00,99.73,2.49,47.97,98.99,SUP_2,86.58,6.32,0.454,11.2,99.08,1.36,1.09
3,LOT-0004,PROD_B,standard,2022-01-06 00:00:00,98.71,2.27,55.95,100.00,SUP_1,87.21,8.63,0.579,15.5,99.80,2.91,1.47
4,LOT-0005,PROD_A,standard,2022-01-06 08:00:00,102.39,2.55,56.20,99.07,SUP_1,87.82,6.91,0.651,18.1,99.61,2.43,1.63


## 2. Build the feature store

In [3]:
features = build_feature_store(tables['laboratory'], tables['process'], tables['deviation_log'])
print(f"Feature store shape: {features.shape}")
feature_cols = get_modeling_columns(features, target='dissolution')
print(f"Modeling features: {len(feature_cols)}")
features[['batch_id', 'dissolution', 'rm_potency', 'compression_force_kn', 'drying_intensity']].describe()

Feature store shape: (1005, 33)
Modeling features: 27


,dissolution,rm_potency,compression_force_kn,drying_intensity
count,1005.000000,1005.000000,1005.000000,1005.000000
mean,83.936965,100.078249,14.922219,7187.794478
std,3.414804,3.028338,1.466220,645.949966
min,70.700000,90.810000,10.460000,5292.300000
25%,82.000000,98.020000,13.960000,6755.130000
50%,84.200000,100.120000,14.870000,7183.200000
75%,86.170000,102.090000,15.830000,7581.910000
max,93.140000,109.180000,19.540000,10109.900000


## 3. Detect OOT/OOS events

In [4]:
detected = detect_oot_events(features, 'dissolution')
suspects = summarize_suspects(detected)
print(f"Total suspect lots: {len(suspects)}")
print(f"Suspect rate: {detected['suspect'].mean():.1%}")
suspects.head(10)

Total suspect lots: 198
Suspect rate: 19.7%


,batch_id,dissolution,_rolling_z,reason_code,severity
0,LOT-0035,77.27,-2.508,OOT-zscore,major
1,LOT-0170,75.72,-3.159,OOT-zscore,major
2,LOT-0282,83.91,-0.322,OOT-run,minor
3,LOT-0283,83.16,-0.611,OOT-run,minor
4,LOT-0284,80.39,-1.695,OOT-run,minor
5,LOT-0285,84.22,-0.072,OOT-run,minor
6,LOT-0286,84.55,0.047,OOT-run,minor
7,LOT-0287,84.02,-0.162,OOT-run,minor
8,LOT-0288,83.36,-0.424,OOT-run,minor
9,LOT-0289,83.89,-0.216,OOT-run,minor


## 4. Peer-batch comparison for a suspect lot

In [5]:
# Pick first suspect lot
target_lot = suspects.iloc[0]['batch_id']
print(f"Investigating lot: {target_lot}")

suspect_row, peers, peer_meta = select_peer_lots(detected, target_lot)
print(f"Peer count: {peer_meta['peer_count']}")
print(f"Low peer warning: {peer_meta['low_peer_warning']}")

comparisons = compute_comparisons(suspect_row, peers, feature_cols)
top_diffs = top_differences(comparisons, n=8)
top_diffs

Investigating lot: LOT-0035
Peer count: 78
Low peer warning: False


,feature,suspect_value,peer_mean,peer_std,z_diff,cohens_d,p_value,direction
0,supplier_SUP_3,1.0000,0.1282,0.3365,2.591,2.591,0.0096,above
1,disintegration,9.3000,15.0179,3.0136,-1.897,-1.897,0.0578,below
2,rm_potency,106.6400,100.7501,3.4278,1.718,1.718,0.0857,above
3,compression_force_kn,12.3200,14.8751,1.5431,-1.656,-1.656,0.0978,below
4,compression_to_weight_ratio,0.0248,0.0297,0.0031,-1.572,-1.572,0.1160,below
5,drying_intensity,6327.5200,7279.1408,648.7606,-1.467,-1.467,0.1424,below
6,content_uniformity_rsd,0.7700,1.9647,0.8252,-1.448,-1.448,0.1477,below
7,drying_temp_c,56.8000,59.6526,2.0065,-1.422,-1.422,0.1551,below


## 5. Rank contributing factors

In [6]:
labels = prepare_labels(features, 'dissolution')
print(f"Label distribution: {labels.value_counts().to_dict()}")

model_result = train_contributor_model(features, feature_cols, labels)
print(f"Primary model CV AUC: {model_result['metrics']['cv_auc_mean']}")
print(f"Benchmark GBM CV AUC: {model_result['benchmark_metrics']['cv_auc_mean']}")

contribs = rank_contributors(model_result, top_n=8)
contribs

Label distribution: {0: 904, 1: 101}


Primary model CV AUC: 0.714
Benchmark GBM CV AUC: 0.644


,feature,coefficient,abs_coefficient,rank,interpretation
0,supplier_SUP_3,0.4660,0.4660,1,higher values increase suspect risk
1,supplier_SUP_1,-0.2679,0.2679,2,lower values increase suspect risk
2,mix_speed_rpm,-0.2629,0.2629,3,lower values increase suspect risk
3,drying_time_min,-0.2616,0.2616,4,lower values increase suspect risk
4,mix_energy_proxy,-0.2369,0.2369,5,lower values increase suspect risk
5,friability,-0.1842,0.1842,6,lower values increase suspect risk
6,mix_time_min,0.1832,0.1832,7,higher values increase suspect risk
7,rm_potency,-0.1768,0.1768,8,lower values increase suspect risk


## 6. Generate investigation memo

In [7]:
output_dir = Path('..') / 'artifacts' / 'reports'
html = generate_investigation_report(
    lot_id=target_lot,
    suspect_row=suspect_row,
    peer_metadata=peer_meta,
    comparisons=top_diffs,
    contributors=contribs,
    output_path=output_dir / f'investigation_{target_lot}.html',
)
print(f"Memo generated: {len(html)} chars")
print(f"Saved to: {output_dir / f'investigation_{target_lot}.html'}")

Memo generated: 5873 chars
Saved to: ../artifacts/reports/investigation_LOT-0035.html


## Verification

Confirm key invariants hold:

In [8]:
assert features['batch_id'].is_unique, 'FAIL: duplicate batch IDs'
assert target_lot not in peers['batch_id'].values, 'FAIL: suspect in own peer set'
assert 'not a determination of root cause' in html, 'FAIL: missing causality caveat'
assert len(comparisons) == len(feature_cols), 'FAIL: comparison row count mismatch'
print('All invariant checks passed.')

All invariant checks passed.
